# Differences from ITensorMPS

Below I outline the main algorithmic differences between ITensorMPOConstruction and ITensorMPS. There are of course lots of optimizations that are more like implementation details that I don't touch on.

## 1. Adaptive construction via the bipartite graph

Instead of returning to the original definition of the Hamiltonian at each site, the current partial MPO construction is used instead. This is essentially the approach from [RenLi2020](https://doi.org/10.1063/5.0018149), but with the minimum vertex cover replaced with a rank decomposition.

Concretely, MPO construction algorithms start by putting the Hamiltonian in bipartite form about the current site:

$$
\mathcal{H} = \sum_{a = 1}^{N_\ell} \sum_{b = 1}^{N_r} \gamma_{a b} A_a \otimes B_b \ .
$$

The existing algorithm in ITensorMPS constructs this bipartite form from the original `OpIDSum`, such that both $A_a$ and $B_b$ are tensor product operators. The key insight of [RenLi2020](https://doi.org/10.1063/5.0018149) was that the left operators $A_a$ can instead be the operators corresponding to the incoming links to the current site, and that instead of going back to the original description of the Hamiltonian, the description can efficiently be updated at each site as the MPO is constructed.

Consider the interaction term of the momentum-space Fermi-Hubbard Hamiltonian

$$
\sum_{p, q, k = 1}^N c^\dagger_{p - k, \uparrow} c^\dagger_{q + k, \downarrow} c_{q, \downarrow} c_{p, \uparrow} \ ,
$$

which can be represented as an MPO of bond dimension $O(N)$. Using the algorithm from ITensorMPS, the coupling matrix $\gamma_{a b}$ is going to have dimensions $O(N^2) \times O(N^2)$, whereas with the adaptive algorithm the dimensions will be $O(N) \times O(N^2)$. See the documentation for [Fermi-Hubbard Hamiltonian in Momentum Space](https://itensor.github.io/ITensorMPOConstruction.jl/dev/examples/fermi-hubbard-ks/).

## 2. Connected components

For Hamiltonians with $U(1)$ symmetries the coupling matrix $\gamma$ can be permuted into block-diagonal form. In ITensorMPS (and I'm pretty sure every other library), this is done by grouping the left and right operators into disjoint sets based on the flux. Even if this operator flux is only calculated for the left operators, $A_a$, efficiently using the precomputed incoming flux and onsite flux, this can take up a large portion of the runtime, especially if the fluxes are then hashed. Basically anything involving an `ITensors.QN` should be as far from a hot loop as possible.

Instead, I realized that $\gamma_{a b}$ can be brought into block-diagonal form by finding the connected components of the bipartite graph defined by $\gamma_{a b}$. The benefits of this are

* There is no need to think about fluxes, and the sparse and dense code-paths are essentially identical.
* This approach guarantees the maximal block-diagonal sparsity regardless of the quantities the end-user wants to conserve.

For example for the 30 site momentum space Fermi-Hubbard Hamiltonian with conservation of total particle number and spin, the MPO produced by ITensorMPS has a sparsity of 92.6%, whereas the MPO from ITensorMPOConstruction is 99.8% sparse because though it is not specified, the underlying Hamiltonian also conserves momentum.

For systems I have looked at, computing the connected components takes up maybe 10% of the total construction time.

## 3. Starting and ending identities

Unlike ITensorMPS, ITensorMPOConstruction does not treat the starting and ending identity strings any differently. While this doesn't have an impact on performance there are a few implications. For one, it makes for cleaner code, and second it means that optimal MPO bond dimensions are produced in every case. My colleague had a use case where he wanted to construct an MPO that was essentially a depth two brickwall circuit of CZ gates, we accomplished this by constructing an MPO for each gate and multiplying them all together. If this was done with ITensorMPS the resulting bond dimension would have grown exponentially (though in all fairness the MPOs from ITensorMPS could be cleanly truncated).

In [31]:
using ITensors, ITensorMPS, ITensorMPOConstruction

sites = siteinds("Qubit", 6)
os = OpSum()
os .+= "I", 3, "I", 4
os .+= "Z", 3, "Z", 4

@show linkdims(MPO(os, sites))
@show linkdims(MPO_new(os, sites))
@show linkdims(truncate(MPO(os, sites); cutoff=1e-30));

linkdims(MPO(os, sites)) = [2, 2, 4, 2, 2]
linkdims(MPO_new(os, sites)) = [1, 1, 2, 1, 1]
linkdims(truncate(MPO(os, sites); cutoff = 1.0e-30)) = [1, 1, 2, 1, 1]


## 4. Sparse QR

Instead of using the SVD to decompose the blocks of $\gamma$, I use the sparse rank revealing QR decomposition provided by SuiteSparse's SPQR. I have experimented with using the SVD and I found SPQR to be much faster, even though if I recall correctly the blocks themselves and the resulting QR factors were fairly dense. Adding in a user option to switch out the matrix decomposition would be simple to add, and is likely worthwhile as there are cases, such as when you want compression, when the SVD is likely superior.

## 5. `OpIDSum`

Instead of using `OpSum` directly, I translate it to a lighter weight and more compact representation (`OpIDSum`) so that I don't have to deal with strings at any point in the main construction loop. More to the point, I avoid interfacing with objects from `ITensors` as much as possible, as they're not designed for this workflow.

## 6. Multithreading

Where possible, I leverage multiple threads to speed up construction. The most obvious example is that I perform the decomposition for each connected component in parallel. Because the construction is largely memory bound (at least for the systems I've seen) the performance gains aren't huge, but a 2x increase is reasonable.